# 07 · Mechanistic interpretability: finding induction heads

**Track:** Training LLMs & Frontier Research · **Runtime (CPU default):** ~10 seconds after the model download · **GPU:** not needed

Show a language model a random sequence of tokens twice. On the first pass it can only guess; on the second pass its
loss collapses, because it has learned to **look back at what came after this token last time, and copy it**. In 2022
Anthropic's interpretability team traced this ability to a specific two-head circuit, the **induction head**: a
*previous-token head* in one layer writes "my previous token was X" into the residual stream, and an *induction head*
in a later layer uses that (via **K-composition**) to find the earlier occurrence of the current token and copy its
successor. They argued induction heads are a major mechanism behind in-context learning. In this notebook we find
that circuit in a real (small) pretrained transformer with **TransformerLens**, then verify it causally with
**ablations** and **activation patching**.

> **Source & license credit.** Built on **[TransformerLens](https://github.com/TransformerLensOrg/TransformerLens)**
> (MIT, v4 `TransformerBridge` API). Model: Neel Nanda's 2-layer attention-only toy model
> **[NeelNanda/Attn_Only_2L512W_C4_Code](https://huggingface.co/NeelNanda/Attn_Only_2L512W_C4_Code)**, downloaded at
> runtime from the Hugging Face Hub (not redistributed; see the model card). The experiment design (repeated random
> tokens, induction/previous-token scores, head ablation and patching) follows Elhage et al. (2021), Olsson et al. (2022)
> and the openly published TransformerLens tutorials. The code here is written fresh. The ARENA curriculum covers the
> same material in depth and is linked rather than copied. See [`research/THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md).

## Learning goals

1. Measure **in-context learning** directly: per-token loss on a repeated random sequence.
2. Read attention patterns and compute **previous-token**, **duplicate-token** and **induction** scores for every head.
3. Use **zero-ablation** to test which heads are *necessary* for the behaviour.
4. Use **activation patching** (clean → corrupted) to test which heads are *sufficient* to restore it.
5. Compute **K-composition scores** from the weights alone and connect them to the circuit.

## Background

* Elhage, Nanda, Olsson, et al., *A Mathematical Framework for Transformer Circuits*, Anthropic 2021.
  [transformer-circuits.pub/2021/framework](https://transformer-circuits.pub/2021/framework/index.html)
* Olsson, Elhage, Nanda, et al., *In-context Learning and Induction Heads*, Anthropic 2022.
  [transformer-circuits.pub/2022/in-context-learning-and-induction-heads](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) · [arXiv:2209.11895](https://arxiv.org/abs/2209.11895)
* Meng et al., *Locating and Editing Factual Associations in GPT* (causal tracing / activation patching), 2022. [arXiv:2202.05262](https://arxiv.org/abs/2202.05262)
* Wang et al., *Interpretability in the Wild: a Circuit for Indirect Object Identification in GPT-2 small*, 2022. [arXiv:2211.00593](https://arxiv.org/abs/2211.00593)
* Heimersheim & Nanda, *How to use and interpret activation patching*, 2024. [arXiv:2404.15255](https://arxiv.org/abs/2404.15255)
* Zhang & Nanda, *Towards Best Practices of Activation Patching in Language Models*, 2023. [arXiv:2309.16042](https://arxiv.org/abs/2309.16042)
* ARENA curriculum, *Transformer Interpretability* chapter (Callum McDougall et al.). [arena.education](https://www.arena.education/) / [GitHub](https://github.com/callummcdougall/ARENA_3.0)

**Why this model?** GPT-2 small (124M) also has induction heads, but it has 144 heads across 12 layers with MLPs in
between, which makes for a slower and messier first exercise. The 2-layer attention-only model (8 heads × 2 layers,
d_model 512; ~50M parameters, most of them in the 48k-token embedding/unembedding) is the smallest *pretrained*
model in which the full circuit exists, and it loads in seconds on CPU. The trade-off: no MLPs, so we only see the
attention part of the story. A GPT-2 small version is included as an optional cell at the end.

In [ ]:
# --- Setup: make research/src importable (from research/, research/notebooks/, or a Colab clone) ---
import sys, pathlib
for _p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]:
    _src = next((c for c in (_p / "src", _p / "research" / "src") if (c / "lab_config.py").exists()), None)
    if _src:
        sys.path.insert(0, str(_src))
        break
else:
    raise RuntimeError("Could not find research/src -- run this notebook from inside the repo.")

from lab_config import CFG, pick, set_seed, heavy_cell, read_text, quiet_hf_logging

FULL_SIZE = CFG.full_size   # <- set True here (or export LAB_FULL_SIZE=1) for the full-size run
CFG.full_size = FULL_SIZE
set_seed()
print(CFG.summary())
quiet_hf_logging()

In [ ]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformer_lens.model_bridge import TransformerBridge

torch.set_grad_enabled(False)
t0 = time.time()
model = TransformerBridge.boot_tl_legacy("NeelNanda/Attn_Only_2L512W_C4_Code", device=CFG.device)
cfg = model.cfg
print(f"loaded in {time.time() - t0:.0f}s: {cfg.n_layers} layers x {cfg.n_heads} heads, d_model={cfg.d_model}, "
      f"d_head={cfg.d_head}, vocab={cfg.d_vocab}, attn_only={cfg.attn_only}")
print("hook points per layer, e.g.:", [k for k in model.hook_dict if k.startswith("blocks.0.attn.hook_")][:6])

## 1 · In-context learning on repeated random tokens

Random tokens carry no statistical pattern a model could have memorised during pre-training, so the only way to
predict the second copy is to look back in context.

In [ ]:
SEQ_LEN = 25
BATCH = pick(8, 32)

def repeated_tokens(batch, seq_len, seed):
    g = torch.Generator().manual_seed(seed)
    rand = torch.randint(1000, 20000, (batch, seq_len), generator=g)   # skip the very rare / special ids
    bos = torch.full((batch, 1), model.tokenizer.bos_token_id)
    return torch.cat([bos, rand, rand], dim=1).to(CFG.device)

def per_token_logprob(logits, tokens):
    """log p(correct next token) at each position: shape [batch, pos-1]."""
    return logits.log_softmax(-1)[:, :-1].gather(-1, tokens[:, 1:, None]).squeeze(-1)

tokens = repeated_tokens(BATCH, SEQ_LEN, CFG.seed)
logits, cache = model.run_with_cache(tokens)
lp = per_token_logprob(logits, tokens)            # position i predicts token i+1
first, second = -lp[:, :SEQ_LEN].mean().item(), -lp[:, SEQ_LEN:].mean().item()
print(f"loss on 1st copy: {first:.2f} nats   loss on 2nd copy: {second:.2f} nats   (uniform = ln V = {np.log(cfg.d_vocab):.2f})")

plt.figure(figsize=(7, 3))
plt.plot(range(1, 2 * SEQ_LEN + 1), -lp.mean(0).cpu(), "o-", ms=3)
plt.axvline(SEQ_LEN + 0.5, ls="--", c="gray"); plt.xlabel("position of predicted token"); plt.ylabel("loss (nats)")
plt.title("Per-token loss on [BOS, x, x]"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 2 · Attention-pattern scores for every head

For a destination (query) position $q$ in the second copy, holding token $x_j$:
* **previous-token head**: attends to $q-1$;
* **duplicate-token head**: attends to the earlier occurrence of the *same* token, offset $\text{SEQ\_LEN}$;
* **induction head**: attends to the token *after* the earlier occurrence, offset $\text{SEQ\_LEN}-1$.

We average the relevant diagonal of each head's attention pattern over the second-copy query positions.

In [ ]:
def diag_score(pattern, offset):
    """pattern: [batch, head, q, k]. Mean attention from q to q-offset, over q in the 2nd copy."""
    d = torch.diagonal(pattern, offset=-offset, dim1=-2, dim2=-1)   # [batch, head, n]
    return d[..., -SEQ_LEN:].mean((0, -1))

scores = {name: torch.zeros(cfg.n_layers, cfg.n_heads) for name in ("previous-token", "duplicate-token", "induction")}
for layer in range(cfg.n_layers):
    pat = cache[f"blocks.{layer}.attn.hook_pattern"]
    scores["previous-token"][layer] = diag_score(pat, 1)
    scores["duplicate-token"][layer] = diag_score(pat, SEQ_LEN)
    scores["induction"][layer] = diag_score(pat, SEQ_LEN - 1)

fig, axes = plt.subplots(1, 3, figsize=(12, 2.6))
for ax, (name, s) in zip(axes, scores.items()):
    im = ax.imshow(s.cpu(), vmin=0, vmax=1, cmap="Blues")
    ax.set(title=f"{name} score", xlabel="head", ylabel="layer", yticks=range(cfg.n_layers), xticks=range(cfg.n_heads))
    for (l, h), v in np.ndenumerate(s.cpu().numpy()):
        ax.text(h, l, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if v > 0.5 else "black")
plt.tight_layout(); plt.show()

PREV_HEAD = (0, int(scores["previous-token"][0].argmax()))
IND_HEAD = (1, int(scores["induction"][1].argmax()))
print(f"strongest previous-token head in layer 0: L{PREV_HEAD[0]}H{PREV_HEAD[1]} ({scores['previous-token'][PREV_HEAD].item():.2f})")
print(f"strongest induction head in layer 1:      L{IND_HEAD[0]}H{IND_HEAD[1]} ({scores['induction'][IND_HEAD].item():.2f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, (layer, head), title in zip(axes, [PREV_HEAD, IND_HEAD], ["previous-token head", "induction head"]):
    ax.imshow(cache[f"blocks.{layer}.attn.hook_pattern"][0, head].cpu(), cmap="viridis")
    ax.set(title=f"L{layer}H{head}: {title}", xlabel="key position", ylabel="query position")
plt.tight_layout(); plt.show()

The previous-token head is a bright line just below the diagonal. The induction head is blank on the first copy and
then a bright line at offset −(SEQ_LEN − 1) on the second copy: each token attends to the token that followed its
previous occurrence.

## 3 · Necessity: zero-ablate each head

We set one head's output (`hook_z`, shape `[batch, pos, head, d_head]`) to zero and measure the loss on the second
copy. Zero-ablation is crude; mean-ablation or resample-ablation are usually preferred because zero can be far
off-distribution (see Heimersheim & Nanda, 2024). Here the effect sizes are large enough that it doesn't matter much.

In [ ]:
def second_copy_loss(logits):
    return -per_token_logprob(logits, tokens)[:, SEQ_LEN:].mean().item()

base_loss = second_copy_loss(logits)
ablation = torch.zeros(cfg.n_layers, cfg.n_heads)
for layer in range(cfg.n_layers):
    for head in range(cfg.n_heads):
        def zero_head(z, hook, head=head):
            z[:, :, head] = 0.0
            return z
        ablation[layer, head] = second_copy_loss(model.run_with_hooks(tokens, fwd_hooks=[(f"blocks.{layer}.attn.hook_z", zero_head)])) - base_loss

plt.figure(figsize=(5, 2.4))
plt.imshow(ablation.cpu(), cmap="Reds"); plt.colorbar(label="Δ loss on 2nd copy")
plt.xlabel("head"); plt.ylabel("layer"); plt.yticks(range(cfg.n_layers)); plt.title("Zero-ablation: increase in loss")
plt.tight_layout(); plt.show()
top = torch.topk(ablation.flatten(), 3)
print("largest effects:", [(f"L{i // cfg.n_heads}H{i % cfg.n_heads}", round(v.item(), 2)) for v, i in zip(top.values, top.indices)])

## 4 · Sufficiency: activation patching (clean → corrupted)

* **Clean run:** `[BOS, x, x]`. The model can do induction.
* **Corrupted run:** `[BOS, y, x]`, where the first copy is replaced by *different* random tokens. Nothing is
  repeated, so induction cannot help on the second half.

For each head we re-run the corrupted input but **patch in that head's clean output** (at all positions), and measure
how much of the clean performance on the second half comes back:
$$\text{recovery} = \frac{\mathcal{M}_{patched} - \mathcal{M}_{corrupt}}{\mathcal{M}_{clean} - \mathcal{M}_{corrupt}},\qquad \mathcal{M} = \text{mean log-prob of the correct 2nd-half tokens.}$$

In [ ]:
g = torch.Generator().manual_seed(CFG.seed + 1)
corrupt = tokens.clone()
corrupt[:, 1:SEQ_LEN + 1] = torch.randint(1000, 20000, (BATCH, SEQ_LEN), generator=g).to(CFG.device)

def metric(logits):
    return per_token_logprob(logits, tokens)[:, SEQ_LEN:].mean().item()   # always score the *clean* targets

clean_logits, clean_cache = model.run_with_cache(tokens)
M_clean, M_corrupt = metric(clean_logits), metric(model(corrupt))
print(f"metric: clean {M_clean:.2f}, corrupted {M_corrupt:.2f}")

def patch_from_clean(layer, head):
    """A hook that overwrites one head's output in the corrupted run with its clean-run value."""
    name = f"blocks.{layer}.attn.hook_z"
    def hook_fn(z, hook):
        z[:, :, head] = clean_cache[name][:, :, head]
        return z
    return (name, hook_fn)

def recovered(*heads):
    patched = metric(model.run_with_hooks(corrupt, fwd_hooks=[patch_from_clean(l, h) for l, h in heads]))
    return (patched - M_corrupt) / (M_clean - M_corrupt)

recovery = torch.zeros(cfg.n_layers, cfg.n_heads)
for layer in range(cfg.n_layers):
    for head in range(cfg.n_heads):
        recovery[layer, head] = recovered((layer, head))

plt.figure(figsize=(5, 2.4))
plt.imshow(recovery.cpu(), cmap="RdBu", vmin=-1, vmax=1); plt.colorbar(label="fraction recovered")
plt.xlabel("head"); plt.ylabel("layer"); plt.yticks(range(cfg.n_layers)); plt.title("Patching clean head output into corrupted run")
plt.tight_layout(); plt.show()
print(f"recovery from patching L{IND_HEAD[0]}H{IND_HEAD[1]} alone: {recovery[IND_HEAD].item():.0%}")
print(f"recovery from patching L{PREV_HEAD[0]}H{PREV_HEAD[1]} alone: {recovery[PREV_HEAD].item():.0%}")

**Why does patching the previous-token head *alone* barely help, even though ablating it hurts a lot?** Its clean
output says "the token before position *k* was $x_{k-1}$". In the corrupted run the induction head can then find the
right *position*, but the token sitting there is now a corrupted $y$, so it copies the wrong thing. The previous-token
head acts **upstream**: its effect on the output flows *through* the induction head (K-composition). Patching the
induction head's output, on the other hand, hands the model the already-copied clean token, which is why it alone
recovers most of the performance. Ablation tells us what is *necessary* (both heads); patching tells us where the
information is *sufficient* (the induction head's output). Single-component patching can miss upstream parts of a
circuit, a central subtlety discussed by Heimersheim & Nanda (2024). Patching both heads together should therefore
add little over patching the induction head alone:

In [ ]:
r_both = recovered(PREV_HEAD, IND_HEAD)
print(f"recovery from patching L{PREV_HEAD[0]}H{PREV_HEAD[1]} + L{IND_HEAD[0]}H{IND_HEAD[1]} together: {r_both:.0%}")

## 5 · Reading the circuit off the weights: K-composition

Elhage et al. (2021) show that an attention head factors into a **QK circuit** $W_{QK} = W_Q W_K^\top$ (where to
attend) and an **OV circuit** $W_{OV} = W_V W_O$ (what to move). A layer-1 head *K-composes* with a layer-0 head when
its keys read what the layer-0 head wrote. In TransformerLens's row-vector convention, that is the product
$W_{QK}^{(1,h')}\,{W_{OV}^{(0,h)}}^{\top}$. The composition score is
$$\frac{\lVert W_{QK}^{(1,h')}\,{W_{OV}^{(0,h)}}^{\top}\rVert_F}{\lVert W_{QK}^{(1,h')}\rVert_F\,\lVert W_{OV}^{(0,h)}\rVert_F}.$$

In [ ]:
W_Q, W_K, W_V, W_O = (getattr(model, n).detach().float().cpu() for n in ("W_Q", "W_K", "W_V", "W_O"))
print("W_Q", tuple(W_Q.shape), "W_O", tuple(W_O.shape))   # [layer, head, d_model, d_head] / [layer, head, d_head, d_model]

def comp_score(A, B):
    return (torch.linalg.matrix_norm(A @ B) / (torch.linalg.matrix_norm(A) * torch.linalg.matrix_norm(B))).item()

k_comp = torch.zeros(cfg.n_heads, cfg.n_heads)   # [layer-1 head, layer-0 head]
for h1 in range(cfg.n_heads):
    W_QK = W_Q[1, h1] @ W_K[1, h1].T
    for h0 in range(cfg.n_heads):
        W_OV = W_V[0, h0] @ W_O[0, h0]
        k_comp[h1, h0] = comp_score(W_QK, W_OV.T)

# baseline: the same score for random matrices with the same shapes / rank
torch.manual_seed(0)
rand = lambda *s: torch.randn(*s)
baseline = np.mean([comp_score(rand(cfg.d_model, cfg.d_head) @ rand(cfg.d_head, cfg.d_model),
                               (rand(cfg.d_model, cfg.d_head) @ rand(cfg.d_head, cfg.d_model)).T) for _ in range(20)])

plt.figure(figsize=(4.5, 3.6))
plt.imshow(k_comp, cmap="Purples"); plt.colorbar()
plt.xlabel("layer-0 head (writer)"); plt.ylabel("layer-1 head (reader)"); plt.title("K-composition scores")
plt.tight_layout(); plt.show()
print(f"K-comp(L1H{IND_HEAD[1]} <- L0H{PREV_HEAD[1]}) = {k_comp[IND_HEAD[1], PREV_HEAD[1]]:.3f};  "
      f"mean over all pairs = {k_comp.mean():.3f};  random baseline ≈ {baseline:.3f}")

## 6 · (Optional, heavier) induction heads in GPT-2 small

GPT-2 small (124M parameters, ~500 MB in fp32) loads fine on most laptops but is slower. Olsson et al. and many
follow-ups report several induction heads in its middle layers. The cell computes the same induction score for all
144 heads. **Skipped unless `FULL_SIZE`.**

In [ ]:
if heavy_cell("Induction-score heatmap for GPT-2 small (124M)"):
    gpt2 = TransformerBridge.boot_transformers("gpt2", device=CFG.device)
    gpt2.enable_compatibility_mode()
    g_tokens = torch.cat([torch.full((8, 1), gpt2.tokenizer.bos_token_id),
                          *(2 * [torch.randint(1000, 20000, (8, SEQ_LEN), generator=torch.Generator().manual_seed(0))])], 1).to(CFG.device)
    _, g_cache = gpt2.run_with_cache(g_tokens, names_filter=lambda n: n.endswith("hook_pattern"))
    g_scores = torch.stack([diag_score(g_cache[f"blocks.{l}.attn.hook_pattern"], SEQ_LEN - 1) for l in range(gpt2.cfg.n_layers)])
    plt.imshow(g_scores.cpu(), cmap="Blues"); plt.colorbar(); plt.xlabel("head"); plt.ylabel("layer"); plt.title("GPT-2 small induction scores"); plt.show()

## Your turn

1. **Mean-ablation.** Replace zero-ablation with the head's mean output over the batch. Do the rankings change?
2. **Position-resolved patching.** Patch the induction head's clean output only at a single query position $q$ in the
   second copy. Which predictions recover? (Only the one at $q$, if the story is right.)
3. **Q- and V-composition.** Compute Q- and V-composition scores ($W_{OV}^{(0)} W_{QK}^{(1)}$ and
   $W_{OV}^{(0)} W_{OV}^{(1)}$). Are they also elevated for the circuit heads, or is it really K-composition?
4. **Natural text.** Find induction behaviour in real text: run the model on a paragraph that repeats a rare name and
   look at the induction head's attention on the second mention.

## Beyond the paper — a mini research question

*When do induction heads form?* Olsson et al. found a **phase change** early in training: induction heads appear
abruptly, at the same moment as a visible bump in the loss curve and a jump in in-context learning score. Using
`research/src/tiny_gpt.py`, train a 2-layer attention-only model (set the MLP to identity) on sequences that mix
natural text with repeated random segments, logging the induction score of every head every 20 steps. Can you
reproduce the phase change at toy scale? How does its timing depend on the fraction of repeated data and on width?
(Reference point: Reddy, *The mechanistic basis of data dependence and abrupt learning in an in-context
classification task*, 2023, [arXiv:2312.03002](https://arxiv.org/abs/2312.03002).)

## References

* Elhage, N. et al. (2021). *A Mathematical Framework for Transformer Circuits.* Anthropic. https://transformer-circuits.pub/2021/framework/index.html
* Olsson, C. et al. (2022). *In-context Learning and Induction Heads.* Anthropic. [arXiv:2209.11895](https://arxiv.org/abs/2209.11895)
* Meng, K. et al. (2022). *Locating and Editing Factual Associations in GPT.* [arXiv:2202.05262](https://arxiv.org/abs/2202.05262)
* Wang, K. et al. (2022). *Interpretability in the Wild (IOI circuit).* [arXiv:2211.00593](https://arxiv.org/abs/2211.00593)
* Heimersheim, S., Nanda, N. (2024). *How to use and interpret activation patching.* [arXiv:2404.15255](https://arxiv.org/abs/2404.15255)
* Zhang, F., Nanda, N. (2023). *Towards Best Practices of Activation Patching.* [arXiv:2309.16042](https://arxiv.org/abs/2309.16042)
* Nanda, N., Bloom, J. et al. *TransformerLens* (MIT). https://github.com/TransformerLensOrg/TransformerLens
* McDougall, C. et al. *ARENA* curriculum. https://github.com/callummcdougall/ARENA_3.0